# Lesson 15 — Presolve and FBBT

## Learning objectives

1. Recognize the standard presolve transformations.
2. Implement feasibility-based bound tightening (FBBT) by hand.
3. Understand probing and its trade-offs.
4. Use `discopt`'s presolve flags to debug bad models.

## 1. Why presolve matters

Presolve runs *before* B&B and aims to reduce problem size, tighten bounds, and detect infeasibility. On real MILPs, presolve commonly reduces variable count 30–60% and constraint count 20–50% {cite:p}`Savelsbergh1994,Achterberg2007`. Time spent in presolve is almost always recovered.

## 2. Standard transformations

| Transformation | Example |
|----------------|---------|
| Singleton row | $x_j \ge 5$ from a single-variable constraint → tighten bound |
| Empty column | Variable with 0 coefficients → fix to bound that helps objective |
| Implied bound | $x_1 + x_2 + x_3 = 1, x_i \in [0,1]$ → already tight |
| Coefficient strengthening | Loose big-M → tighten via implied bounds |
| Aggregation | $x_1 = 2 x_2 + 3$ → eliminate $x_1$ |
| Probing | Set $x_j = 0$, propagate; if infeasible, fix $x_j = 1$ |

See {cite:p}`Savelsbergh1994` and {cite:p}`Belotti2009` for the systematic catalogue.

## 3. Feasibility-based bound tightening (FBBT)

Given a constraint $\sum_j a_j x_j \le b$ with $x_j \in [\ell_j, u_j]$, the implied bound on each $x_k$ is

$$x_k \le \frac{b - \sum_{j \ne k, a_j > 0} a_j \ell_j - \sum_{j \ne k, a_j < 0} a_j u_j}{a_k} \quad \text{(if } a_k > 0\text{)}.$$

(Mirror for $a_k < 0$ and for $\ge$ constraints.) Apply to every variable in every constraint; iterate to a fixed point. {cite:p}`Belotti2009` formalize this for the MINLP setting.

## 4. Probing

Pick a binary $x_j$. Tentatively set $x_j = 0$, run FBBT. If infeasible, fix $x_j = 1$. Repeat for $x_j = 1$. Either reduces variables or generates *implied bounds* and *cliques* ($x_j = 1 \Rightarrow x_k = 0$). Quadratic in the number of binaries; usually limited to a budget {cite:p}`Achterberg2007`.

### Presolve in `discopt`

There is no standalone `m.presolve()` entry point — presolve runs *inside*
`m.solve()`. The orchestrator (in the Rust backend, `discopt-core/src/presolve/orchestrator.rs`)
drives ~20 named passes (FBBT, OBBT, aggregation, implied bounds,
coefficient strengthening, probing, clique detection, redundancy
elimination, scaling, symmetry, polynomial factorable elimination, ...) to
a *fixed point* under a global time/work budget. The full pass catalogue
lives under `crates/discopt-core/src/presolve/`; see also
`crates/discopt-core/src/presolve/ROADMAP.md` for status.

The Python-visible knobs on `m.solve()` are:

| Argument | Default | What it does |
|---|---|---|
| `presolve=True` | on | master switch for the root-presolve pipeline |
| `presolve_polynomial=False` | off | enables the (slower) polynomial / factorable elimination passes |
| `presolve_reverse_ad=False` | off | reverse-AD-based bound tightening on top of FBBT |
| `in_tree_presolve_stride=0` | off | run light presolve every `k` B&B nodes (0 = root only) |

The headline effect of presolve is visible without any flag: a smaller
`r.node_count` and faster `r.wall_time`. Exercises 2 and 5 below let you
build FBBT yourself on a single constraint and *measure* the per-instance
speedup of toggling `presolve=False`.

## 5. FBBT in MINLP

For nonlinear constraints, FBBT uses **interval arithmetic** {cite:p}`Moore1966,Neumaier1990`. Given $z = x \cdot y$ with $x \in [a, b], y \in [c, d]$, the implied bound is $z \in [\min(ac, ad, bc, bd), \max(...)]$.

This yields tight enclosures for many problems but can be loose for high-rank monomials. **Optimization-based BT (OBBT)** instead solves a small auxiliary LP/NLP to compute each bound — much tighter, much slower.

## 6. Effects on B&B

Tighter bounds → tighter LP relaxation → more pruning → smaller tree. Plus tighter bounds → tighter big-M and McCormick relaxations → tighter convex envelopes.

This is why "pre-modelling" (giving good initial bounds in your formulation) is usually the highest-leverage thing you can do for solver performance.

## References
{cite:p}`Savelsbergh1994` (LP/MILP), {cite:p}`Belotti2009` (MINLP), {cite:p}`Achterberg2007`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm
